In [1]:
import google.cloud.bigquery as bq
import pandas_gbq
import pandas as pd
import numpy as np

In [2]:
project_id = 'mudah-analytics-222509'
date_query = '2021-05-29'

In [ ]:
blocket = pd.read_gbq(
    '''
        select distinct pay_log_id, DATE(paid_at) AS date
        from blocket.pay_log
        #where format_date("%Y",paid_at) >='2021'
        order by date desc
    
    ''', project_id = project_id)

# # client = bigquery.Client()
# # df_bq = client.query(blocket).to_dataframe()
# bq_assistant.estimate_query_size(blocket)


In [ ]:
blocket_stream = pd.read_gbq(
    '''
        select distinct pay_log_id, DATE(paid_at) AS date
        from blocket_stream.pay_log
        where format_date("%Y",paid_at) ='2021'
        order by date desc
    
    ''', project_id = project_id)

In [ ]:
blocket_stream_dbaction = pd.read_gbq(
    '''
        select distinct pay_log_id, DATE(publish_time) AS date
        from blocket_stream.pay_log_dbactions
        where format_date("%Y",publish_time) ='2021'
        order by date desc
    
    ''', project_id = project_id)

In [ ]:
print('-'*80)

In [ ]:
left = blocket.loc[blocket['date'] == date_query]
left

In [ ]:
right = blocket_stream.loc[blocket_stream['date'] == date_query]
right

In [ ]:
missing = pd.merge(left, right, how = 'outer', indicator = 'Exists')

In [ ]:
print('-'*80)

In [ ]:
extra_id = missing.loc[missing['Exists'] == 'left_only']
extra_id.head(5)

In [ ]:
print('-'*80)

In [ ]:
still_missing = missing.loc[missing['Exists'] == 'right_only']
still_missing.head(5)

In [ ]:
print('-'*80)

In [ ]:
deleted_or_not = pd.merge(still_missing, blocket_stream_dbaction, how = 'left', indicator = 'Deleted_Check')
still_missing.head(5)

In [ ]:
print('-'*80)

In [ ]:
within_date = pd.merge(extra_id, blocket_stream, how = 'left', indicator = 'Still_Exist')
within_date

In [ ]:
print('-'*80)

In [ ]:
print ("Id's in (blocket)                              : " ,blocket['pay_log_id'].nunique())
print ("Id's in (blocket_stream)                       : " ,blocket_stream['pay_log_id'].nunique())
print ('Difference between blocket and blocket_stream  : ', blocket_stream['pay_log_id'].nunique()-blocket['pay_log_id'].nunique())
print('-'*80)
print ("Ids found in (blocket) and found deleted       : " ,deleted_or_not['Deleted_Check'].count())
print ("Ids still missing in (blocket_stream)          : " ,within_date['Still_Exist'].count())